# Clase 5 — Evaluación con Golden Set

*Arquitectura de Aplicaciones con IA Generativa — EscuelaIT · 90 min · bloques 3 y 4 (32 min)*

**Objetivo del notebook**: entender qué es una evaluación de app LLM y construir, en vivo, un mini sistema de eval con golden set + métricas de retrieval + LLM-as-judge.

**Flujo**

- §1–§6: **conceptos** (tablas, métricas, ejemplos). 12 min.
- §7–§14: **demo ejecutable** (mini corpus + retriever + golden set + judge + comparación de prompts). 20 min.

Los conceptos vienen primero porque sin ellos los números no significan nada.

<style>
.jp-RenderedMarkdown { font-size: 1.15em; line-height: 1.5; }
.jp-RenderedMarkdown h1 { font-size: 2.1em; }
.jp-RenderedMarkdown h2 { font-size: 1.6em; margin-top: 0.6em; }
.jp-RenderedMarkdown h3 { font-size: 1.25em; }
.jp-RenderedMarkdown table { font-size: 0.95em; }
.jp-RenderedMarkdown th, .jp-RenderedMarkdown td { padding: 6px 10px; }
.jp-RenderedMarkdown pre { font-size: 0.9em; }
</style>

## §1 — La pregunta que nadie hace hasta que duele

> *"Cambié el system prompt y ahora el cliente dice que el bot responde peor. ¿Cómo lo demuestro o lo desmiento?"*

Sin evaluación, esa pregunta no tiene respuesta — solo opiniones.

| Decisión típica en una app LLM | Sin eval | Con eval |
|---|---|---|
| Cambiar de modelo (GPT-4 → DeepSeek) | *"a mí me parece igual"* | precision@k cae 8 pts → no migrar todavía |
| Reescribir el system prompt | esperar quejas del usuario | LLM-as-judge sube de 3.2 a 4.1 → mergear |
| Cambiar chunk size 800 → 400 | rezar | recall@5 sube, faithfulness baja → tradeoff explícito |
| Activar reranking | añadir latencia *por si acaso* | MRR pasa de 0.62 a 0.81 → vale los 200 ms extra |

**Regla**: si no puedes medir el efecto de un cambio, no estás iterando — estás adivinando.

## §2 — Tres niveles de evaluación

Una app RAG falla en sitios distintos. Mezclar todo en "¿la respuesta es buena?" oculta el verdadero culpable.

| Nivel | Qué se mide | Pregunta que responde | Métricas típicas |
|---|---|---|---|
| **1. Retrieval** | El retriever trajo los chunks correctos | ¿Tengo la información para responder? | precision@k, recall@k, MRR, hit rate |
| **2. Generación** | El LLM usa bien el contexto que le di | ¿La respuesta es fiel a los chunks? | faithfulness, citations correctness |
| **3. End-to-end** | El usuario recibe lo que pidió | ¿Resolvió la consulta? | LLM-as-judge global, satisfaction, task success |

Si la calidad cae, **siempre se diagnostica de abajo hacia arriba**: primero retrieval, luego generación, luego end-to-end. Si retrieval falla, mejorar el prompt no arregla nada.

## §3 — Golden set: el contrato de qué es "bueno"

> Un golden set es un dataset pequeño, **estable** y **representativo** de queries reales con la respuesta esperada (o el chunk esperado).

Es un **contrato**: define qué significa "funciona" antes de tocar nada. Cuando lo modificas, lo modificas con intención (nuevo caso de uso) — no a posteriori para tapar una regresión.

### Propiedades de un buen golden set

| Propiedad | Por qué importa | Anti-pattern |
|---|---|---|
| **Estable** | Permite comparar versión A vs B en el tiempo | regenerarlo cada sprint con queries nuevas |
| **Representativo** | Cubre los tipos reales de query que verás | solo casos felices |
| **Curado a mano** | La verdad de referencia es **humana** | generado por el mismo LLM que evalúas |
| **Diverso en dificultad** | Casos fáciles + difíciles + adversariales | solo preguntas que ya sabes que funcionan |
| **Pequeño** | 20-100 entradas alcanza, debe correr en minutos | 10 000 queries → nadie corre la eval |

### Ejemplo de entrada de golden set para una app RAG

```python
{
  "query": "¿Cada cuánto se monitorea la calidad del agua en pozos de relaves?",
  "relevant_doc_ids": ["DS-024-EM"],          # qué docs DEBEN aparecer
  "reference_answer": "Mensualmente, según...", # respuesta humana de referencia
  "difficulty": "easy",                       # tag opcional para análisis
}
```

## §4 — Offline vs Online

| Tipo | Cuándo corre | Para qué sirve | Cuándo lo necesitas |
|---|---|---|---|
| **Offline** (golden set) | en tu máquina o CI, contra dataset fijo | gate antes de mergear cambios | desde el día 1 |
| **Online** (sampleo de prod) | sobre tráfico real anonimizado | descubrir queries que tu golden set no cubre | a partir de tener usuarios |

Hoy nos centramos en **offline**. Online merece un módulo aparte (anonimización + consentimiento + storage).

## §5 — Métricas de retrieval

Para una query con conjunto de docs **relevantes** $R$ y un retriever que devuelve top-$k$ docs $D_k$:

| Métrica | Fórmula | Qué mide | Útil cuando |
|---|---|---|---|
| **precision@k** | $\frac{|R \cap D_k|}{k}$ | qué fracción del top-k es relevante | el LLM lee todos los k chunks |
| **recall@k** | $\frac{|R \cap D_k|}{|R|}$ | qué fracción de relevantes recuperaste | hay múltiples docs relevantes |
| **hit rate@k** | $1$ si $|R \cap D_k|>0$ else $0$ | ¿al menos uno relevante? | tarea binaria "¿hay info?" |
| **MRR** | $\frac{1}{\text{rank del primer relevante}}$ | qué tan arriba aparece el primer relevante | el orden importa (cita-1) |

### Ejemplo intuitivo

```
Query: "¿factor de seguridad ICOLD?"
Relevantes: {ICOLD-194}
Top-5:      [ICOLD-194, GISTM, DS-024, Panel, ANA]    ← MRR=1/1=1.0, P@5=1/5=0.20, R@5=1/1=1.0
Top-5:      [GISTM, DS-024, ICOLD-194, Panel, ANA]    ← MRR=1/3=0.33
```

**No hay una métrica única**: precisión y recall se mueven en direcciones opuestas con $k$. Reportas las dos, o una compuesta como $F_1$.

## §6 — LLM-as-judge

Para evaluar la **calidad** de una respuesta generada (no solo si recuperaste bien), usamos un LLM como evaluador con una rúbrica explícita.

### Rúbrica típica para RAG

| Dimensión | Pregunta | Escala |
|---|---|---|
| **Faithfulness** | ¿La respuesta usa solo info de los chunks recuperados? | 1–5 |
| **Relevance** | ¿Responde lo que se preguntó? | 1–5 |
| **Completeness** | ¿Cubre lo de la respuesta de referencia? | 1–5 |
| **Citations** | ¿Las citas apuntan a chunks reales y soportan la afirmación? | 1–5 |

### Sesgos conocidos del LLM-as-judge

| Sesgo | Síntoma | Mitigación |
|---|---|---|
| **Position bias** | prefiere la primera respuesta presentada | aleatorizar orden A/B |
| **Length bias** | premia respuestas largas | penalizar verbosidad en la rúbrica |
| **Self-preference** | un modelo se puntúa más alto a sí mismo | usar un modelo *distinto* al que generó |
| **Verbosidad del judge** | reasoning excesivo no se traduce a mejor score | pedir score numérico estructurado |

**Regla**: el LLM-as-judge **no reemplaza** al humano — es un proxy escalable. Calibras periódicamente con un humano en una muestra y verificas que correlaciona.

---

## DEMO ejecutable

Construimos en vivo:

1. Mini corpus inspirado en el dominio de `yana-killa-demo` (relaves mineros, hidrogeología).
2. Retriever didáctico (MiniLM + numpy, sin DB — el patrón de clase 3 simplificado).
3. Golden set de 8 preguntas curadas a mano.
4. Métricas de retrieval (precision@k, recall@k, MRR).
5. Generación con DeepSeek + citas.
6. LLM-as-judge con rúbrica (faithfulness, relevance, completeness).
7. **Comparación de dos system prompts** (estricto vs relajado) → así se ve un gate de regresión.

---

## §7 — Setup

In [ ]:
import os, json, re
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer

load_dotenv()  # busca .env hacia arriba — toma DEEPSEEK_API_KEY del raíz del curso

client = OpenAI(
    api_key=os.environ['DEEPSEEK_API_KEY'],
    base_url='https://api.deepseek.com',
)

print('DeepSeek client listo. Modelo por defecto: deepseek-chat')

In [ ]:
def chat(messages, model='deepseek-chat', temperature=0):
    """Wrapper minimal sobre el client. temperature=0 para resultados reproducibles en eval."""
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return resp.choices[0].message.content

# Smoke test
chat([{'role': 'user', 'content': 'Responde solo "ok".'}])

## §8 — Mini corpus

8 chunks de 6 documentos sintéticos. Inspirados en el dominio del demo (normativa minera + estándares internacionales) pero **escritos para esta clase**, no extraídos del corpus real.

Cada chunk lleva: `chunk_id`, `doc_id` (su "documento" lógico), `page` y `text`.

In [ ]:
CORPUS = [
    {
        'chunk_id': 'c1',
        'doc_id': 'DS-024-EM',
        'page': 12,
        'text': (
            'El monitoreo de calidad del agua en pozos de monitoreo asociados a depósitos '
            'de relaves se realiza con frecuencia mensual durante la operación. Los parámetros '
            'incluyen pH, conductividad, metales totales y sulfatos. La frecuencia trimestral '
            'aplica solamente en fase post-cierre cuando los indicadores se han estabilizado.'
        ),
    },
    {
        'chunk_id': 'c2',
        'doc_id': 'DS-024-EM',
        'page': 15,
        'text': (
            'En caso de detectarse excedencias en parámetros críticos, la frecuencia mensual '
            'se intensifica a quincenal y se notifica a la autoridad competente en un plazo '
            'no mayor a 72 horas.'
        ),
    },
    {
        'chunk_id': 'c3',
        'doc_id': 'ICOLD-194',
        'page': 47,
        'text': (
            'ICOLD Bulletin 194 recomienda un factor de seguridad mínimo de 1.5 para '
            'condiciones estáticas y de 1.1 para condiciones sísmicas en taludes de presas '
            'de relaves. Estos valores asumen análisis con métodos de equilibrio límite.'
        ),
    },
    {
        'chunk_id': 'c4',
        'doc_id': 'GISTM',
        'page': 8,
        'text': (
            'El Global Industry Standard on Tailings Management exige una estructura de '
            'gobernanza con un Independent Tailings Review Board (ITRB) compuesto por '
            'expertos externos a la organización operadora, con autoridad para auditar '
            'las decisiones técnicas críticas.'
        ),
    },
    {
        'chunk_id': 'c5',
        'doc_id': 'GISTM',
        'page': 22,
        'text': (
            'GISTM establece factores de seguridad equivalentes a los de ICOLD para '
            'condiciones estáticas, pero introduce el concepto de consequence classification '
            'donde presas de mayor consecuencia exigen FS sísmico hasta 1.2.'
        ),
    },
    {
        'chunk_id': 'c6',
        'doc_id': 'Panel-Feijao',
        'page': 55,
        'text': (
            'El Expert Panel sobre la falla de Feijão en Brumadinho concluyó que la causa '
            'principal fue licuación estática asociada a una pérdida progresiva de '
            'resistencia en relaves no drenados, agravada por geometría desfavorable y '
            'monitoreo piezométrico insuficiente.'
        ),
    },
    {
        'chunk_id': 'c7',
        'doc_id': 'DS-040-EM',
        'page': 31,
        'text': (
            'Durante el cierre de mina, las labores de revegetación deben iniciarse en la '
            'fase de cierre progresivo, no esperando al cierre final. Se priorizan especies '
            'nativas y se monitorea la cobertura vegetal por al menos 5 años post-cierre.'
        ),
    },
    {
        'chunk_id': 'c8',
        'doc_id': 'ANA-2016',
        'page': 4,
        'text': (
            'La Autoridad Nacional del Agua (ANA) es la entidad competente para emitir '
            'autorizaciones de uso de agua subterránea en el Perú. La autorización requiere '
            'estudio hidrogeológico previo y monitoreo durante la vigencia del permiso.'
        ),
    },
]

print(f'Corpus: {len(CORPUS)} chunks de {len({c["doc_id"] for c in CORPUS})} documentos')

## §9 — Embedder + retriever

Mismo patrón de Clase 3: MiniLM multilingüe, similitud coseno, top-K. Sin DB — todo en memoria, que es lo que cabe para 8 chunks.

In [ ]:
embedder = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Pre-computar embeddings del corpus (con norma 1 → coseno = producto punto)
corpus_embs = embedder.encode(
    [c['text'] for c in CORPUS],
    normalize_embeddings=True,
)
print(f'Embeddings: shape={corpus_embs.shape}')

In [ ]:
def retrieve(query: str, k: int = 3):
    """Top-k chunks por similitud coseno. Devuelve la lista enriquecida con `score`."""
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    sims = corpus_embs @ q_emb  # producto punto = coseno (norma 1)
    top_idx = np.argsort(-sims)[:k]
    return [{**CORPUS[i], 'score': float(sims[i])} for i in top_idx]

# Test rápido
for hit in retrieve('¿factor de seguridad sísmico?', k=3):
    print(f"{hit['score']:.3f}  [{hit['doc_id']} p.{hit['page']}]  {hit['text'][:80]}…")

## §10 — Golden set

8 preguntas curadas a mano. Mezcla intencional de:

- **Fáciles**: respuesta literal en un solo doc.
- **Multi-doc**: respuesta requiere 2+ docs (comparativas).
- **Adversariales**: pregunta cuya respuesta NO está en el corpus → la app debería decir *"no sé"* en lugar de inventar.

Para cada pregunta listamos los `relevant_doc_ids` (qué docs DEBEN aparecer) y una `reference_answer` humana.

In [ ]:
GOLDEN_SET = [
    {
        'qid': 'q1',
        'query': '¿Con qué frecuencia se monitorea la calidad del agua en pozos de relaves durante la operación?',
        'relevant_doc_ids': ['DS-024-EM'],
        'reference_answer': 'Mensualmente durante la operación. La frecuencia trimestral aplica solo en post-cierre.',
        'difficulty': 'easy',
    },
    {
        'qid': 'q2',
        'query': '¿Cuál es el factor de seguridad mínimo recomendado por ICOLD para condición sísmica?',
        'relevant_doc_ids': ['ICOLD-194'],
        'reference_answer': '1.1 para condición sísmica, según ICOLD Bulletin 194.',
        'difficulty': 'easy',
    },
    {
        'qid': 'q3',
        'query': '¿Qué exige GISTM en términos de gobernanza independiente?',
        'relevant_doc_ids': ['GISTM'],
        'reference_answer': 'Un Independent Tailings Review Board (ITRB) con expertos externos y autoridad sobre decisiones técnicas críticas.',
        'difficulty': 'easy',
    },
    {
        'qid': 'q4',
        'query': '¿Cuál fue la causa principal de la falla en Brumadinho?',
        'relevant_doc_ids': ['Panel-Feijao'],
        'reference_answer': 'Licuación estática por pérdida de resistencia en relaves no drenados, con monitoreo piezométrico insuficiente.',
        'difficulty': 'easy',
    },
    {
        'qid': 'q5',
        'query': '¿Cuándo deben iniciar las labores de revegetación en el cierre de mina?',
        'relevant_doc_ids': ['DS-040-EM'],
        'reference_answer': 'En la fase de cierre progresivo, no esperando al cierre final.',
        'difficulty': 'easy',
    },
    {
        'qid': 'q6',
        'query': '¿Cómo se compara el factor de seguridad sísmico de ICOLD con el de GISTM?',
        'relevant_doc_ids': ['ICOLD-194', 'GISTM'],
        'reference_answer': 'ICOLD recomienda 1.1 sísmico; GISTM iguala estática pero exige hasta 1.2 sísmico para presas de alta consecuencia.',
        'difficulty': 'multi-doc',
    },
    {
        'qid': 'q7',
        'query': '¿Qué autoridad emite autorizaciones de uso de agua subterránea en Perú?',
        'relevant_doc_ids': ['ANA-2016'],
        'reference_answer': 'La Autoridad Nacional del Agua (ANA), previo estudio hidrogeológico.',
        'difficulty': 'easy',
    },
    {
        'qid': 'q8',
        'query': '¿Cuál es la altura máxima permitida para una presa de relaves según el GISTM?',
        'relevant_doc_ids': [],  # ← respuesta NO está en el corpus
        'reference_answer': 'NO_DISPONIBLE — el corpus no especifica altura máxima.',
        'difficulty': 'adversarial',
    },
]
print(f'Golden set: {len(GOLDEN_SET)} queries')

## §11 — Métricas de retrieval

Para cada query corremos el retriever y comparamos los `doc_id`s del top-K contra los `relevant_doc_ids`.

Notar: para queries adversariales (`relevant_doc_ids = []`) las métricas no aplican — se filtran del agregado.

In [ ]:
def precision_at_k(retrieved_docs, relevant, k):
    if k == 0:
        return 0.0
    top_k = retrieved_docs[:k]
    return sum(1 for d in top_k if d in relevant) / k

def recall_at_k(retrieved_docs, relevant, k):
    if not relevant:
        return None  # no aplica para adversariales
    top_k = retrieved_docs[:k]
    return sum(1 for d in top_k if d in relevant) / len(relevant)

def reciprocal_rank(retrieved_docs, relevant):
    if not relevant:
        return None
    for i, d in enumerate(retrieved_docs, start=1):
        if d in relevant:
            return 1.0 / i
    return 0.0

In [ ]:
K = 3
rows = []
for entry in GOLDEN_SET:
    hits = retrieve(entry['query'], k=K)
    retrieved_docs = [h['doc_id'] for h in hits]
    relevant = set(entry['relevant_doc_ids'])
    rows.append({
        'qid': entry['qid'],
        'difficulty': entry['difficulty'],
        'top1': retrieved_docs[0] if retrieved_docs else None,
        f'P@{K}': precision_at_k(retrieved_docs, relevant, K),
        f'R@{K}': recall_at_k(retrieved_docs, relevant, K),
        'MRR': reciprocal_rank(retrieved_docs, relevant),
    })

# Render simple
print(f"{'qid':<5} {'dif':<12} {'top1':<14} {'P@K':<6} {'R@K':<6} {'MRR':<6}")
for r in rows:
    p = f"{r[f'P@{K}']:.2f}"
    rr = '—' if r[f'R@{K}'] is None else f"{r[f'R@{K}']:.2f}"
    mr = '—' if r['MRR'] is None else f"{r['MRR']:.2f}"
    print(f"{r['qid']:<5} {r['difficulty']:<12} {(r['top1'] or '—'):<14} {p:<6} {rr:<6} {mr:<6}")

In [ ]:
# Agregado (excluyendo adversariales)
scored = [r for r in rows if r['MRR'] is not None]
p_avg = np.mean([r[f'P@{K}'] for r in scored])
r_avg = np.mean([r[f'R@{K}'] for r in scored])
mrr_avg = np.mean([r['MRR'] for r in scored])

print(f'Sobre {len(scored)} queries no-adversariales:')
print(f'  Precision@{K} promedio: {p_avg:.2f}')
print(f'  Recall@{K} promedio:    {r_avg:.2f}')
print(f'  MRR:                   {mrr_avg:.2f}')

## §12 — Generación con dos system prompts

El paso 2 es generar la respuesta. Aquí montamos dos versiones del **mismo flujo** que solo difieren en el `system prompt`. Esa es exactamente la situación de "cambié el prompt, ¿está mejor o peor?".

- **`prompt_strict`**: obliga al modelo a citar y a decir *no sé* si no está en el contexto.
- **`prompt_loose`**: tono libre, sin restricciones explícitas.

Esperamos que `loose` se vea bien a ojo pero saque peores scores en faithfulness y en queries adversariales — esa es la **regresión silenciosa** que la eval atrapa.

In [ ]:
PROMPT_STRICT = (
    'Eres un asistente técnico que responde preguntas sobre normativa minera y estándares de relaves.\n'
    'REGLAS:\n'
    '1. Responde SOLO con información que aparezca explícitamente en los fragmentos provistos.\n'
    '2. Si la información no está en los fragmentos, responde exactamente: NO_DISPONIBLE.\n'
    '3. Cita los fragmentos usados con el formato [doc_id p.X].\n'
    '4. Sé conciso (máximo 3 frases).'
)

PROMPT_LOOSE = (
    'Eres un asistente experto en minería y normativa ambiental. '
    'Responde la pregunta del usuario de la mejor manera posible usando los fragmentos como referencia.'
)

def build_user_message(query, hits):
    fragments = '\n\n'.join(
        f"[{h['doc_id']} p.{h['page']}] {h['text']}" for h in hits
    )
    return f'FRAGMENTOS:\n{fragments}\n\nPREGUNTA: {query}'

def answer(query, system_prompt, k=3):
    hits = retrieve(query, k=k)
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': build_user_message(query, hits)},
    ]
    return chat(messages), hits

# Probar una
ans, hits = answer(GOLDEN_SET[0]['query'], PROMPT_STRICT)
print('PREGUNTA:', GOLDEN_SET[0]['query'])
print('RESPUESTA:', ans)

## §13 — LLM-as-judge

Un LLM separado puntúa la respuesta contra: la pregunta, los fragmentos recuperados, y la respuesta de referencia. Devuelve JSON con scores 1-5 en tres dimensiones.

Notas:

- `temperature=0` para que el judge sea reproducible.
- Pedimos JSON estructurado **dentro** del prompt (sin `response_format` para mantenerlo agnóstico de proveedor).
- El judge **no ve qué prompt generó la respuesta** → evita sesgo de simpatía.

In [ ]:
JUDGE_PROMPT = (
    'Eres un evaluador estricto de respuestas de un asistente RAG. '
    'Recibes: la pregunta, los fragmentos recuperados, la respuesta del asistente y una respuesta de referencia. '
    'Devuelves SOLO un JSON con tres scores 1-5 (5 = excelente):\n'
    '- faithfulness: ¿la respuesta usa solo info de los fragmentos? (5 = totalmente fiel; 1 = inventa)\n'
    '- relevance:    ¿responde lo preguntado? (5 = sí, directo; 1 = se va por las ramas)\n'
    '- completeness: ¿cubre lo de la respuesta de referencia? (5 = sí; 1 = omite lo central)\n'
    'Caso especial: si la pregunta no se puede responder con los fragmentos y el asistente respondió NO_DISPONIBLE, '
    'asigna faithfulness=5, relevance=5, completeness=5.\n'
    'Formato exacto: {"faithfulness": N, "relevance": N, "completeness": N}'
)

def judge(query, hits, answer_text, reference):
    fragments = '\n'.join(f"[{h['doc_id']} p.{h['page']}] {h['text'][:200]}" for h in hits)
    user_msg = (
        f'PREGUNTA: {query}\n\n'
        f'FRAGMENTOS:\n{fragments}\n\n'
        f'RESPUESTA DEL ASISTENTE:\n{answer_text}\n\n'
        f'RESPUESTA DE REFERENCIA:\n{reference}\n\n'
        'JSON:'
    )
    raw = chat(
        [{'role': 'system', 'content': JUDGE_PROMPT}, {'role': 'user', 'content': user_msg}],
        temperature=0,
    )
    # Parser tolerante: extrae el primer { ... }
    m = re.search(r'\{[^}]+\}', raw)
    if not m:
        return {'faithfulness': 0, 'relevance': 0, 'completeness': 0, 'parse_error': True}
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return {'faithfulness': 0, 'relevance': 0, 'completeness': 0, 'parse_error': True}

# Test del judge
ans, hits = answer(GOLDEN_SET[0]['query'], PROMPT_STRICT)
judge(GOLDEN_SET[0]['query'], hits, ans, GOLDEN_SET[0]['reference_answer'])

## §14 — Eval comparativa: STRICT vs LOOSE

Corremos las 8 queries con cada prompt, dejando el judge igual. Esto tarda ~1-2 min porque son ~32 llamadas (8 queries × 2 prompts × 2 calls = generación + judge).

Salida: tabla por query y agregado. La regresión esperada está en la query adversarial (q8) y posiblemente en faithfulness.

In [ ]:
def run_eval(system_prompt, label):
    results = []
    for entry in GOLDEN_SET:
        ans, hits = answer(entry['query'], system_prompt)
        scores = judge(entry['query'], hits, ans, entry['reference_answer'])
        results.append({
            'qid': entry['qid'],
            'difficulty': entry['difficulty'],
            'answer': ans,
            **{k: scores.get(k, 0) for k in ['faithfulness', 'relevance', 'completeness']},
        })
        print(f"  [{label}] {entry['qid']}: F={scores.get('faithfulness',0)} R={scores.get('relevance',0)} C={scores.get('completeness',0)}")
    return results

print('Corriendo eval con PROMPT_STRICT...')
results_strict = run_eval(PROMPT_STRICT, 'STRICT')

print('\nCorriendo eval con PROMPT_LOOSE...')
results_loose = run_eval(PROMPT_LOOSE, 'LOOSE')

In [ ]:
def aggregate(results, label):
    f = np.mean([r['faithfulness'] for r in results])
    rel = np.mean([r['relevance'] for r in results])
    c = np.mean([r['completeness'] for r in results])
    overall = (f + rel + c) / 3
    print(f'  {label:<8}  F={f:.2f}  R={rel:.2f}  C={c:.2f}  overall={overall:.2f}')
    return overall

print('=' * 50)
print('AGREGADO sobre 8 queries del golden set')
print('=' * 50)
o_strict = aggregate(results_strict, 'STRICT')
o_loose  = aggregate(results_loose,  'LOOSE')

delta = o_loose - o_strict
verdict = 'mejora' if delta > 0.1 else ('empeora' if delta < -0.1 else 'similar')
print(f'\nVeredicto: cambiar a LOOSE → score global {verdict} ({delta:+.2f}).')

## §15 — Inspeccionar el caso adversarial (q8)

Es donde la regresión silenciosa suele esconderse: la pregunta cuya respuesta **no está** en el corpus. Un modelo bien guiado dice *NO_DISPONIBLE*. Un modelo libre **inventa**.

In [ ]:
for label, results in [('STRICT', results_strict), ('LOOSE', results_loose)]:
    r = next(x for x in results if x['qid'] == 'q8')
    print(f'\n=== {label} en q8 (adversarial) ===')
    print('Faithfulness:', r['faithfulness'])
    print('Respuesta:', r['answer'][:300])

## §16 — De aquí a CI/CD

Lo que acabamos de hacer manualmente es lo que se automatiza:

```
# .github/workflows/eval.yml (esquemático)
on: pull_request
jobs:
  eval:
    steps:
      - run: python eval/run.py --golden golden_set.json --output report.json
      - run: python eval/gate.py --report report.json --baseline baseline.json --threshold 0.1
```

**Reglas razonables del gate**:

| Regla | Acción si falla |
|---|---|
| `overall_drop > 0.3 puntos` | bloquear merge |
| `faithfulness_drop > 0.2 puntos` | bloquear merge (riesgo de alucinación) |
| `precision@k_drop > 0.05` | bloquear si el cambio toca retrieval o chunking |
| Cualquier query nueva en el set | aviso, no bloqueo |

**Cadencia y coste**:

- Eval offline (golden set 50-100 queries): cada PR que toque prompts/retrieval/modelo.
- Eval online (sampleo de prod): semanal o diaria sobre tráfico anonimizado.
- Coste típico de un golden set de 50 queries con LLM-as-judge: 0.10-0.50 USD por corrida.

**Trampa habitual**: dejar el golden set congelado dos años → eventualmente prueba lo que ya no se usa. Revisa cada trimestre.

## §17 — Tres ideas para llevarse

1. **Sin golden set, no estás iterando — estás adivinando.** El golden set es el contrato de qué significa *bueno* antes de tocar nada.
2. **Evalúa por niveles.** Retrieval, generación y end-to-end fallan en sitios distintos. Si lo mezclas todo, no sabes a quién culpar.
3. **LLM-as-judge es un proxy, no un humano.** Calibra contra humano periódicamente; usa rúbrica explícita; no uses el mismo modelo que generó.

### Cierre del curso

| Clase | Idea de cierre |
|---|---|
| 1 — Fundamentos y ecosistema | El LLM es **infraestructura, no feature**. |
| 2 — Prompt engineering | Los prompts son **código**, no arte oculto. |
| 3 — RAG | RAG = búsqueda semántica + inyección de contexto. **Lo que no mides, no mejoras.** |
| 4 — Construcción de la app | El borde (auth, rate limit, modos de fallo) separa demo de producto. |
| 5 — Despliegue y producción | **Decisiones**, no plantillas. Y **medida** continua. |